# 基于一致性哈希环的分布式哈希表模拟

分布式哈希表需要把大量key稳定地分配到多个物理节点，并在节点加入或离开时尽量减少数据迁移。普通取模分片实现简单，但节点数量变化会改变取模基数，使大量key重新映射；一致性哈希则把key和节点映射到同一个环形哈希空间，通过顺时针后继规则完成路由，把节点变更的影响限制在局部环区间。

本实验基于C++17和CMake实现普通取模分片与一致性哈希分片的对照模拟。实验生成一组固定key，使用确定性哈希函数计算位置，分别观察节点加入、节点删除时的迁移比例，并对比不同虚拟节点数量下的节点负载、标准差和不均衡率。

本节学习大纲如下：

1. 实验概述：介绍实验目标、前置知识和实验要点；
2. 环境准备：创建工程目录并检查C++17/CMake工具链；
3. 问题分析：分析稳定哈希、取模路由、一致性哈希环、虚拟节点和节点变更流程；
4. 核心程序开发：实现稳定哈希、哈希环、负载指标、迁移统计和实验主流程；
5. 结果验证与性能分析：编译运行工程，读取迁移比例和负载摘要并进行对比；
6. 实验总结：归纳一致性哈希降低迁移范围和虚拟节点改善负载均衡的作用。


---
## 1. 实验概述

本实验以分布式哈希表的节点路由为问题背景，比较普通取模分片和一致性哈希分片在节点加入、节点删除场景下的数据迁移差异。实验还改变每个物理节点对应的虚拟节点数量，观察虚拟节点对负载均衡的影响。

工程使用确定性的32位FNV-1a加avalanche混合函数，避免`std::hash`在不同编译器或运行环境下的结果差异。每个物理节点通过`node#序号`生成多个虚拟节点，所有虚拟节点和key都映射到同一个`[0, 2^32)`哈希环上。


### 1.1 实验目标

完成本实验后应达到以下目标：

1. 理解分布式哈希表中的key、哈希函数、物理节点、虚拟节点和路由规则之间的组织关系，认识节点数量变化对key归属的影响。
2. 掌握取模分片和一致性哈希环的核心算法，能够完成物理节点到虚拟节点的映射、key的顺时针后继查找，以及节点加入和删除时的迁移统计流程。
3. 具备根据迁移比例、节点负载、标准差和不均衡率比较分片策略的能力，并能够解释虚拟节点数量变化对负载分布的影响。


### 1.2 前置知识

本实验要求提前具备以下基础：

1. 哈希表基础：理解哈希函数、哈希值、槽位和冲突处理的基本概念。
2. 分布式分片基础：理解key如何根据路由规则分配到不同物理节点，以及节点扩缩容带来的数据迁移问题。
3. C++与CMake基础：能够阅读C++17工程，了解头文件、源文件、CMake构建配置和Shell运行脚本之间的关系。
4. 排序与二分查找基础：理解有序数组、`lower_bound`和环形回绕查找的基本用法。
5. 统计指标基础：理解最小负载、最大负载、平均负载、标准差和不均衡率的含义。


### 1.3 实验要点

实验中应重点关注以下内容：

1. 稳定哈希：相同字符串在不同运行中必须得到相同哈希值，保证实验结果可复现；
2. 取模对照：节点数量变化会改变取模基数，观察大量key重新映射的现象；
3. 哈希环路由：节点和key使用同一哈希空间，key归属顺时针方向遇到的第一个虚拟节点所属的物理节点；
4. 虚拟节点：同一物理节点映射多个环位置，观察更细粒度区间对负载均衡的改善；
5. 结果分析：同时查看迁移比例和负载不均衡率，不能只用单一指标评价策略。


---
## 2. 环境准备

### 2.1 创建实验目录并检查编译环境

本实验只依赖C++17标准库和CMake，不依赖CANN、HCCL或NPU。Notebook会在`src/03.02_extra_consistent_hash_table`目录下生成一份可独立编译运行的工程。

目录划分如下：

- `include/`：保存稳定哈希、哈希环、指标和实验接口；
- `src/`：保存哈希函数、哈希环、统计指标、实验流程和主程序；
- `scripts/`：保存自动配置、编译和运行脚本；
- `results/`：保存Notebook运行输出。


In [ ]:
from pathlib import Path
import os
import subprocess

WORK_DIR = Path("src/03.02_extra_consistent_hash_table").resolve()
for directory in [WORK_DIR / "include", WORK_DIR / "src", WORK_DIR / "scripts", WORK_DIR / "results"]:
    directory.mkdir(parents=True, exist_ok=True)

print("Experiment directory:", WORK_DIR)
print("Current platform:", os.name)
subprocess.run(["c++", "--version"], check=False)
subprocess.run(["cmake", "--version"], check=False)


### 2.2 写入工程公共接口

本节写入稳定哈希、哈希环、指标和实验流程的公共头文件。实现文件将在第4节写入，CMake构建文件和运行脚本将在第5节写入。运行到第5节后，Notebook生成的目录将具备完整的C++17工程结构。


In [ ]:
%%writefile src/03.02_extra_consistent_hash_table/include/stable_hash.hpp
#pragma once

#include <cstdint>
#include <string>
#include <vector>

namespace consistent_hash {

// 32 位哈希值的取值空间大小，用于把哈希环理解为 [0, 2^32) 的环形区间。
constexpr std::uint64_t kHashSpace = 1ULL << 32;

// 对相同字符串始终返回相同的 32 位哈希值，保证不同实验运行之间可复现。
std::uint32_t stable_hash(const std::string& value);

// 使用固定种子生成一组可复现且名称唯一的实验键。
std::vector<std::string> generate_keys(int num_keys, std::uint32_t seed);

// 将键序号和伪随机盐值格式化为便于观察的键名。
std::string format_key(int index, std::uint32_t salt);

}  // namespace consistent_hash


In [ ]:
%%writefile src/03.02_extra_consistent_hash_table/include/consistent_hash_ring.hpp
#pragma once

#include <cstdint>
#include <map>
#include <string>
#include <utility>
#include <vector>

namespace consistent_hash {

// 一致性哈希环：把物理节点扩展为多个虚拟节点并映射到 32 位哈希空间。
class ConsistentHashRing {
public:
    // virtual_nodes 表示每个物理节点对应的虚拟节点数量。
    ConsistentHashRing(std::vector<std::string> nodes, int virtual_nodes);

    // 将键映射到其哈希位置顺时针方向遇到的第一个虚拟节点所属的物理节点。
    std::string get_node(const std::string& key) const;

    // 批量计算 key -> physical node 的路由结果。
    std::map<std::string, std::string> assign_keys(const std::vector<std::string>& keys) const;

    // 返回排序后的前 limit 个环位置，用于打印和观察哈希环结构。
    std::vector<std::pair<std::uint32_t, std::string>> describe_ring(int limit) const;

private:
    // 根据物理节点列表和虚拟节点数构造有序哈希环。
    void build_ring();

    std::vector<std::string> nodes_; // 参与路由的物理节点名称。
    int virtual_nodes_; // 每个物理节点创建的虚拟节点数量。
    std::vector<std::pair<std::uint32_t, std::string>> ring_; // 按哈希位置升序保存 (virtual node position, physical node) 记录。
    std::vector<std::uint32_t> positions_; // ring_ 中哈希位置的平行数组，用于 lower_bound 二分查找。
};

// 传统取模路由基线：hash(key) % node_count，供迁移比例对照实验使用。
std::map<std::string, std::string> assign_by_modulo(
    const std::vector<std::string>& keys,
    const std::vector<std::string>& nodes);

}  // namespace consistent_hash


In [ ]:
%%writefile src/03.02_extra_consistent_hash_table/include/metrics.hpp
#pragma once

#include <map>
#include <string>
#include <vector>

namespace consistent_hash {

// 一次键分配结果的节点负载统计摘要。
struct LoadSummary {
    int min_load = 0;
    int max_load = 0;
    double avg_load = 0.0;
    // 节点负载的总体标准差，越小表示分布越均匀。
    double std_load = 0.0;
    // 最大负载 / 平均负载，理想值接近 1。
    double imbalance_ratio = 0.0;
};

// 一种路由策略在一次节点变更事件中的数据迁移统计。
struct MigrationRow {
    std::string strategy;
    std::string event;
    std::string virtual_nodes;
    int migrated = 0;
    // 发生归属变化的键数量占总键数的比例。
    double ratio = 0.0;
};

// 根据 key -> node 分配结果计算各节点的最小、最大、平均负载等指标。
LoadSummary summarize_load(
    const std::map<std::string, std::string>& assignments,
    const std::vector<std::string>& nodes);

// 对比节点变更前后的键归属，返回迁移键数量及迁移比例。
std::pair<int, double> migration_ratio(
    const std::map<std::string, std::string>& before,
    const std::map<std::string, std::string>& after);

// 以下函数负责将指标格式化为终端实验表格。
void print_load_summary(const std::string& label, const LoadSummary& summary);

void print_migration_table(const std::vector<MigrationRow>& rows);

std::string format_percent(double ratio);

}  // namespace consistent_hash


In [ ]:
%%writefile src/03.02_extra_consistent_hash_table/include/experiment.hpp
#pragma once

#include "metrics.hpp"

#include <cstdint>
#include <map>
#include <string>
#include <vector>

namespace consistent_hash {

// 实验输入参数：控制键规模、初始节点数、虚拟节点对比组和随机数据。
struct ExperimentConfig {
    int num_keys = 10000;
    int num_base_nodes = 4;
    std::vector<int> virtual_node_options = {1, 5, 20, 100}; // 依次用不同虚拟节点数量构造哈希环，观察迁移比例和负载均衡变化。
    std::uint32_t seed = 2026;
    int ring_observation_limit = 12; // 最多打印多少个有序虚拟节点位置。
};

// 汇总取模法和一致性哈希法的全部实验结果
struct ExperimentResult {
    std::vector<MigrationRow> migration_rows;
    std::map<std::string, LoadSummary> modulo_summaries; // 场景名称(before/after_add/after_remove) -> 取模法负载摘要。
    std::map<int, std::map<std::string, LoadSummary>> consistent_summaries; // 虚拟节点数 -> 场景名称 -> 一致性哈希负载摘要。
    std::vector<std::pair<std::uint32_t, std::string>> ring_positions; // 用于观察哈希环的部分 (position, physical node) 记录。
};

// 执行基线分配、节点加入/删除、迁移统计和负载统计的完整实验。
ExperimentResult run_experiment(const ExperimentConfig& config);

void print_experiment(
    const ExperimentConfig& config,
    const ExperimentResult& result,
    const std::vector<std::string>& base_nodes,
    const std::string& added_node,
    const std::string& removed_node);

// 生成 node_0、node_1 等初始物理节点名称。
std::vector<std::string> make_base_nodes(int num_base_nodes);

// 将形如 "1,5,20,100" 的命令行文本解析为虚拟节点数量列表。
std::vector<int> parse_virtual_nodes(const std::string& text);

}  // namespace consistent_hash


### 2.3 工程公共接口检查

公共头文件写入完成后，检查关键文件是否已经生成。检查结果均为`OK`时，说明实验输入、哈希环和指标接口已经就绪。


In [ ]:
required_headers = [
    "include/stable_hash.hpp",
    "include/consistent_hash_ring.hpp",
    "include/metrics.hpp",
    "include/experiment.hpp",
]

for relative_path in required_headers:
    path = WORK_DIR / relative_path
    print(f"{relative_path}: {'OK' if path.exists() else 'MISSING'}")


---
## 3. 问题分析

本节分析稳定哈希、取模分片、一致性哈希环、虚拟节点、节点动态变化和实验指标。后续C++实现将围绕这些数据结构与算法展开。


### 3.1 稳定哈希与实验key

实验使用确定性的32位FNV-1a哈希，并增加avalanche混合步骤，使输入字符串的微小变化能够影响更多输出位。哈希空间理解为环形区间：

$$
H=[0,2^{32})
$$

`generate_keys`使用固定种子和线性同余状态生成可复现的盐值，再将序号和盐值格式化为`key_000000_******`形式的唯一key。所有策略使用同一批key和同一批初始节点，保证对照实验只受路由策略影响。


### 3.2 取模分片

取模路由直接使用哈希值对物理节点数量取模：

$$
nodeIndex=hash(key)\bmod N
$$

当节点数从$N$变为$N+1$，大部分key的取模结果都可能变化；当删除节点后，剩余节点数量变为$N-1$，同样会导致大量key重新计算归属。因此取模方法实现简单，但节点动态变化时迁移比例通常较高。


### 3.3 一致性哈希环与顺时针后继查找

一致性哈希把每个虚拟节点和key映射到同一个环形空间。对key计算哈希位置后，在按位置升序保存的虚拟节点数组中使用`lower_bound`查找第一个不小于key位置的虚拟节点：

1. 如果找到位置，key归属该虚拟节点对应的物理节点；
2. 如果key位置大于所有虚拟节点位置，则回绕到数组第一个位置；
3. 节点变更只会影响新增节点或删除节点相邻的环形区间。

工程同时保存`ring_`和`positions_`两个数组：`ring_`保存“虚拟节点位置→物理节点”的完整记录，`positions_`是平行的位置数组，用于快速二分查找。


### 3.4 虚拟节点

一个物理节点可以映射为多个虚拟节点，例如物理节点`node_0`在虚拟节点数为5时生成：

```text
node_0#0, node_0#1, node_0#2, node_0#3, node_0#4
```

`#序号`会参与哈希输入，因此每个虚拟节点通常落在不同环位置，但路由时保存的仍然是对应的物理节点名称。虚拟节点数量增加后，一个物理节点负责的环区间被切成更多小区间并与其他节点交错分布，通常可以降低负载标准差和不均衡率；但迁移比例还会受到具体环位置影响，不要求随虚拟节点数量单调变化。


### 3.5 节点加入、删除与迁移统计

实验分别建立节点变更前、节点加入后和节点删除后的路由状态。对同一key比较变更前后的物理节点归属，如果节点不同，就计为一次迁移：

$$
migrationRatio=\frac{migratedKeys}{totalKeys}
$$

新增节点场景把`node_4`加入初始`node_0`至`node_3`的集合；删除节点场景从原始集合中删除`node_1`。两个场景都与原始状态比较，不构成连续的“先新增再删除”过程。


### 3.6 负载指标

对每个物理节点统计承载key数量，计算：

- `min_load`：最小节点负载；
- `max_load`：最大节点负载；
- `avg_load`：平均节点负载；
- `std_load`：负载总体标准差；
- `imbalance_ratio`：最大负载与平均负载之比。

理想情况下，`std_load`接近0，`imbalance_ratio`接近1。迁移比例反映节点变更成本，负载指标反映静态分布效果，两者需要结合分析。


### 3.7 实验参数设置

默认实验生成10000个key，初始物理节点数为4，虚拟节点配置为`1,5,20,100`，随机种子为2026，并打印前12个哈希环位置。下面先计算理论上的平均负载和哈希空间信息。


In [ ]:
NUM_KEYS = 10000
BASE_NODES = 4
VIRTUAL_NODE_OPTIONS = [1, 5, 20, 100]
SEED = 2026
RING_LIMIT = 12
HASH_SPACE = 2**32

print("numKeys:", NUM_KEYS)
print("baseNodes:", BASE_NODES)
print("virtualNodeOptions:", VIRTUAL_NODE_OPTIONS)
print("seed:", SEED)
print("hash space:", f"[0, {HASH_SPACE})")
print("average load before change:", NUM_KEYS / BASE_NODES)
for virtual_nodes in VIRTUAL_NODE_OPTIONS:
    print(
        f"virtual_nodes={virtual_nodes}: "
        f"ring positions={BASE_NODES * virtual_nodes}"
    )


需要注意，虚拟节点数量增加会扩大哈希环上的位置数量和构建、查询数据结构规模，但不会改变key数量。实验应同时观察环位置数量、迁移比例和负载均衡指标，避免把“虚拟节点越多越好”作为无条件结论。


---
## 4. 核心程序开发

本节依次实现稳定哈希与key生成、哈希环路由、负载指标、迁移统计、实验场景构造和主程序。所有实现均与交付版C++工程保持一致。


### 4.1 稳定哈希与key生成

`stable_hash`不使用实现相关的`std::hash`，而是固定使用32位FNV-1a和avalanche混合；`generate_keys`使用固定种子生成唯一key集合，为不同策略提供完全相同的输入。


In [ ]:
%%writefile src/03.02_extra_consistent_hash_table/src/stable_hash.cpp
#include "stable_hash.hpp"

#include <iomanip>
#include <sstream>
#include <stdexcept>

namespace consistent_hash {

namespace {

// 线性同余生成器：只用于生成可复现实验键中的盐值，不承担哈希环路由。
std::uint32_t next_lcg(std::uint32_t state) {
    return state * 1664525u + 1013904223u;
}

}  // namespace

std::uint32_t stable_hash(const std::string& value) {
    // 先执行 32 位 FNV-1a，再通过 avalanche 混合增强高低位扩散效果。
    // 不使用 std::hash 是为了保证不同编译器和运行环境下的实验结果稳定可复现。
    std::uint32_t hash = 2166136261u;
    for (unsigned char ch : value) {
        hash ^= ch;
        hash *= 16777619u;
    }
    // avalanche 阶段让输入中的微小变化尽可能影响更多输出位。
    hash ^= hash >> 16u;
    hash *= 0x7feb352du;
    hash ^= hash >> 15u;
    hash *= 0x846ca68bu;
    hash ^= hash >> 16u;
    return hash;
}

std::string format_key(int index, std::uint32_t salt) {
    if (index < 0) {
        throw std::invalid_argument("key index must be non-negative");
    }
    // index 保证键名唯一，salt 改变字符串内容以避免键位置呈简单递增规律。
    std::ostringstream out;
    out << "key_" << std::setw(6) << std::setfill('0') << index << "_"
        << std::setw(6) << std::setfill('0') << (salt % 1000000u);
    return out.str();
}

std::vector<std::string> generate_keys(int num_keys, std::uint32_t seed) {
    if (num_keys <= 0) {
        throw std::invalid_argument("num_keys must be positive");
    }

    std::vector<std::string> keys;
    keys.reserve(static_cast<std::size_t>(num_keys));

    // 固定 seed 会生成完全相同的键集合，便于公平比较不同路由策略。
    std::uint32_t state = seed;
    for (int i = 0; i < num_keys; ++i) {
        state = next_lcg(state);
        keys.push_back(format_key(i, state));
    }
    return keys;
}

}  // namespace consistent_hash


### 4.2 一致性哈希环构建与路由

`build_ring`为每个物理节点创建指定数量的虚拟节点，计算虚拟节点哈希位置后排序，并同步构造位置数组。`get_node`使用`lower_bound`完成key到顺时针后继虚拟节点的定位，再返回对应的物理节点名称。`assign_by_modulo`提供取模分片基线。


In [ ]:
%%writefile src/03.02_extra_consistent_hash_table/src/consistent_hash_ring.cpp
#include "consistent_hash_ring.hpp"

#include "stable_hash.hpp"

#include <algorithm>
#include <stdexcept>

namespace consistent_hash {

ConsistentHashRing::ConsistentHashRing(std::vector<std::string> nodes, int virtual_nodes)
    : nodes_(std::move(nodes)), virtual_nodes_(virtual_nodes) {
    // 空节点集合无法路由，虚拟节点数也必须为正数。
    if (nodes_.empty()) {
        throw std::invalid_argument("node list must not be empty");
    }
    if (virtual_nodes_ <= 0) {
        throw std::invalid_argument("virtual_nodes must be positive");
    }
    build_ring();
}

void ConsistentHashRing::build_ring() {
    ring_.clear();
    positions_.clear();
    ring_.reserve(nodes_.size() * static_cast<std::size_t>(virtual_nodes_));

    for (const auto& node : nodes_) {
        for (int vnode = 0; vnode < virtual_nodes_; ++vnode) {
            // 同一物理节点通过 node#序号 生成多个互不相同的虚拟节点位置。
            const std::string vnode_name = node + "#" + std::to_string(vnode);
            ring_.push_back({stable_hash(vnode_name), node});
        }
    }

    // 哈希环用升序数组表示；哈希碰撞时按节点名排序以保证结果确定。
    std::sort(ring_.begin(), ring_.end(), [](const auto& left, const auto& right) {
        if (left.first == right.first) {
            return left.second < right.second;
        }
        return left.first < right.first;
    });

    // 单独保存位置数组，使查询时可以直接使用 lower_bound 做二分查找。
    positions_.reserve(ring_.size());
    for (const auto& item : ring_) {
        positions_.push_back(item.first);
    }
}

std::string ConsistentHashRing::get_node(const std::string& key) const {
    const std::uint32_t key_position = stable_hash(key);
    auto it = std::lower_bound(positions_.begin(), positions_.end(), key_position); // 找到第一个 position >= key_position 的虚拟节点，即沿环顺时针遇到的节点。
    std::size_t index = static_cast<std::size_t>(it - positions_.begin());
    if (index == positions_.size()) {
        index = 0; // 键落在最大虚拟节点之后时回绕到环起点。
    }
    return ring_[index].second;
}

std::map<std::string, std::string> ConsistentHashRing::assign_keys(
    const std::vector<std::string>& keys) const {
    std::map<std::string, std::string> assignments;
    // map 使结果按键名稳定排序，也便于变更前后按同一个 key 比较归属。
    for (const auto& key : keys) {
        assignments.emplace(key, get_node(key));
    }
    return assignments;
}

std::vector<std::pair<std::uint32_t, std::string>> ConsistentHashRing::describe_ring(
    int limit) const {
    // 非法或过大的 limit 统一解释为返回完整哈希环。
    if (limit <= 0 || static_cast<std::size_t>(limit) > ring_.size()) {
        limit = static_cast<int>(ring_.size());
    }
    return {ring_.begin(), ring_.begin() + limit};
}

std::map<std::string, std::string> assign_by_modulo(
    const std::vector<std::string>& keys,
    const std::vector<std::string>& nodes) {
    if (nodes.empty()) {
        throw std::invalid_argument("node list must not be empty");
    }

    std::map<std::string, std::string> assignments;
    for (const auto& key : keys) {
        // 节点数量改变会改变取模基数，因此大量键的节点下标会随之变化。
        const auto index = static_cast<std::size_t>(stable_hash(key) % nodes.size());
        assignments.emplace(key, nodes[index]);
    }
    return assignments;
}

}  // namespace consistent_hash


### 4.3 负载摘要与迁移比例

`summarize_load`先累计每个物理节点承载的key数量，再计算最小值、最大值、平均值、总体标准差和不均衡率。`migration_ratio`通过比较变更前后的两个`map<key,node>`，统计归属发生变化的key数量和比例。


In [ ]:
%%writefile src/03.02_extra_consistent_hash_table/src/metrics.cpp
#include "metrics.hpp"

#include <cmath>
#include <iomanip>
#include <iostream>
#include <map>
#include <sstream>
#include <stdexcept>

namespace consistent_hash {

LoadSummary summarize_load(
    const std::map<std::string, std::string>& assignments,
    const std::vector<std::string>& nodes) {
    if (nodes.empty()) {
        throw std::invalid_argument("node list must not be empty");
    }

    // 先按分配结果累计每个节点实际持有的键数量。
    std::map<std::string, int> counter;
    for (const auto& [key, node] : assignments) {
        (void)key;
        ++counter[node];
    }

    // 按 nodes 列表取负载，确保没有分到键的节点也以 0 计入统计。
    std::vector<int> loads;
    loads.reserve(nodes.size());
    for (const auto& node : nodes) {
        loads.push_back(counter[node]);
    }

    // 一次扫描得到最小值、最大值和总量。
    int min_load = loads.front();
    int max_load = loads.front();
    double total = 0.0;
    for (int load : loads) {
        min_load = std::min(min_load, load);
        max_load = std::max(max_load, load);
        total += load;
    }

    const double avg = total / static_cast<double>(loads.size());
    // 这里使用总体方差（除以节点数），描述整组节点负载的离散程度。
    double variance = 0.0;
    for (int load : loads) {
        const double delta = static_cast<double>(load) - avg;
        variance += delta * delta;
    }
    variance /= static_cast<double>(loads.size());

    LoadSummary summary;
    summary.min_load = min_load;
    summary.max_load = max_load;
    summary.avg_load = avg;
    summary.std_load = std::sqrt(variance);
    // 最大负载与平均负载越接近，imbalance_ratio 越接近 1。
    summary.imbalance_ratio = avg == 0.0 ? 0.0 : static_cast<double>(max_load) / avg;
    return summary;
}

std::pair<int, double> migration_ratio(
    const std::map<std::string, std::string>& before,
    const std::map<std::string, std::string>& after) {
    if (before.size() != after.size()) {
        throw std::invalid_argument("assignment maps must have the same size");
    }

    // 同一个 key 在变更前后归属节点不同，就计为一次数据迁移。
    int migrated = 0;
    for (const auto& [key, before_node] : before) {
        auto it = after.find(key);
        if (it == after.end()) {
            throw std::invalid_argument("assignment maps must contain the same keys");
        }
        if (before_node != it->second) {
            ++migrated;
        }
    }

    const double ratio = before.empty() ? 0.0 : static_cast<double>(migrated) / before.size();
    return {migrated, ratio};
}

std::string format_percent(double ratio) {
    // 内部比例使用 [0, 1] 小数，展示时转换为百分数并保留两位小数。
    std::ostringstream out;
    out << std::fixed << std::setprecision(2) << ratio * 100.0 << "%";
    return out.str();
}

void print_load_summary(const std::string& label, const LoadSummary& summary) {
    std::cout << std::left << std::setw(28) << label
              << " min=" << std::right << std::setw(5) << summary.min_load
              << " max=" << std::setw(5) << summary.max_load
              << " avg=" << std::setw(8) << std::fixed << std::setprecision(2) << summary.avg_load
              << " std=" << std::setw(8) << std::fixed << std::setprecision(2) << summary.std_load
              << " imbalance=" << std::setw(6) << std::fixed << std::setprecision(3)
              << summary.imbalance_ratio << '\n';
}

void print_migration_table(const std::vector<MigrationRow>& rows) {
    // 将取模法和不同虚拟节点配置的一致性哈希结果放在同一张表中对比。
    std::cout << "\nMigration ratio comparison\n";
    std::cout << std::string(78, '-') << '\n';
    std::cout << std::left << std::setw(24) << "strategy"
              << std::setw(12) << "event"
              << std::right << std::setw(14) << "virtual_nodes"
              << std::setw(10) << "migrated"
              << std::setw(10) << "ratio" << '\n';
    std::cout << std::string(78, '-') << '\n';

    for (const auto& row : rows) {
        std::cout << std::left << std::setw(24) << row.strategy
                  << std::setw(12) << row.event
                  << std::right << std::setw(14) << row.virtual_nodes
                  << std::setw(10) << row.migrated
                  << std::setw(10) << format_percent(row.ratio) << '\n';
    }
    std::cout << std::string(78, '-') << '\n';
}

}  // namespace consistent_hash


### 4.4 实验场景与对照流程

`run_experiment`为所有策略生成同一批key和节点集合，分别执行取模基线和不同虚拟节点配置的一致性哈希实验。每种配置都建立三张环：原始环、加入节点后的环和删除节点后的环，然后统计迁移比例和负载摘要。


In [ ]:
%%writefile src/03.02_extra_consistent_hash_table/src/experiment.cpp
#include "experiment.hpp"

#include "consistent_hash_ring.hpp"
#include "stable_hash.hpp"

#include <algorithm>
#include <iostream>
#include <sstream>
#include <stdexcept>

namespace consistent_hash {

std::vector<std::string> make_base_nodes(int num_base_nodes) {
    if (num_base_nodes < 2) {
        throw std::invalid_argument("num_base_nodes must be at least 2");
    }
    // 使用稳定名称保证相同配置下的虚拟节点哈希位置可复现。
    std::vector<std::string> nodes;
    nodes.reserve(static_cast<std::size_t>(num_base_nodes));
    for (int i = 0; i < num_base_nodes; ++i) {
        nodes.push_back("node_" + std::to_string(i));
    }
    return nodes;
}

std::vector<int> parse_virtual_nodes(const std::string& text) {
    // 逐个解析逗号分隔值，空片段跳过，非正数直接拒绝。
    std::vector<int> values;
    std::stringstream input(text);
    std::string token;
    while (std::getline(input, token, ',')) {
        if (token.empty()) {
            continue;
        }
        int value = std::stoi(token);
        if (value <= 0) {
            throw std::invalid_argument("virtual node counts must be positive");
        }
        values.push_back(value);
    }
    if (values.empty()) {
        throw std::invalid_argument("at least one virtual node count is required");
    }
    return values;
}

ExperimentResult run_experiment(const ExperimentConfig& config) {
    // 所有策略共享同一批键和初始节点，确保对比只受路由策略影响。

    // 步骤一：生成关键字集合并计算确定性哈希值。
    const auto keys = generate_keys(config.num_keys, config.seed);
    const auto base_nodes = make_base_nodes(config.num_base_nodes);
    const auto added_node = "node_" + std::to_string(config.num_base_nodes);
    const auto removed_node = base_nodes[1];

    // 分别构造“新增一个节点”和“删除一个原节点”两种独立场景。
    std::vector<std::string> add_nodes = base_nodes;
    add_nodes.push_back(added_node);
    std::vector<std::string> remove_nodes;
    std::copy_if(base_nodes.begin(), base_nodes.end(), std::back_inserter(remove_nodes),
                 [&removed_node](const std::string& node) { return node != removed_node; });

    ExperimentResult result;

    // 步骤二：执行取模分片对照实验。
    const auto modulo_before = assign_by_modulo(keys, base_nodes);
    const auto modulo_after_add = assign_by_modulo(keys, add_nodes);
    const auto modulo_after_remove = assign_by_modulo(keys, remove_nodes);

    // 两种变更都与原始节点状态比较，而不是先新增再删除的连续过程。
    auto [mod_add_migrated, mod_add_ratio] = migration_ratio(modulo_before, modulo_after_add);
    auto [mod_remove_migrated, mod_remove_ratio] = migration_ratio(modulo_before, modulo_after_remove);

    result.migration_rows.push_back(
        {"modulo", "add", "-", mod_add_migrated, mod_add_ratio});
    result.migration_rows.push_back(
        {"modulo", "remove", "-", mod_remove_migrated, mod_remove_ratio});

    // 同时记录变更前后各节点的负载均衡指标。
    result.modulo_summaries["before"] = summarize_load(modulo_before, base_nodes);
    result.modulo_summaries["after_add"] = summarize_load(modulo_after_add, add_nodes);
    result.modulo_summaries["after_remove"] = summarize_load(modulo_after_remove, remove_nodes);

    // 步骤三：执行一致性哈希分片实验。
    for (int virtual_nodes : config.virtual_node_options) {
        // 每一种虚拟节点配置都重新构造三张环，分别对应原始、新增和删除场景。
        ConsistentHashRing before_ring(base_nodes, virtual_nodes);
        ConsistentHashRing add_ring(add_nodes, virtual_nodes);
        ConsistentHashRing remove_ring(remove_nodes, virtual_nodes);

        // 把同一批键路由到三张环，再逐键比较迁移并统计负载。
        const auto before = before_ring.assign_keys(keys);
        const auto after_add = add_ring.assign_keys(keys);
        const auto after_remove = remove_ring.assign_keys(keys);

        auto [add_migrated, add_ratio] = migration_ratio(before, after_add);
        auto [remove_migrated, remove_ratio] = migration_ratio(before, after_remove);

        result.migration_rows.push_back({"consistent_hash",
                                         "add",
                                         std::to_string(virtual_nodes),
                                         add_migrated,
                                         add_ratio});
        result.migration_rows.push_back({"consistent_hash",
                                         "remove",
                                         std::to_string(virtual_nodes),
                                         remove_migrated,
                                         remove_ratio});

        // 步骤四：负载均衡分析。
        result.consistent_summaries[virtual_nodes]["before"] =
            summarize_load(before, base_nodes);
        result.consistent_summaries[virtual_nodes]["after_add"] =
            summarize_load(after_add, add_nodes);
        result.consistent_summaries[virtual_nodes]["after_remove"] =
            summarize_load(after_remove, remove_nodes);
    }

    // 使用第一组虚拟节点配置截取部分有序位置，帮助观察哈希环数据结构。
    ConsistentHashRing observation_ring(base_nodes, config.virtual_node_options.front());
    result.ring_positions = observation_ring.describe_ring(config.ring_observation_limit);

    return result;
}

// 步骤五：实验结果对比。
void print_experiment(
    const ExperimentConfig& config,
    const ExperimentResult& result,
    const std::vector<std::string>& base_nodes,
    const std::string& added_node,
    const std::string& removed_node) {
    // 按“配置、迁移比例、负载摘要、环位置”的顺序输出实验结果。
    std::cout << "Experiment configuration\n";
    std::cout << std::string(78, '-') << '\n';
    std::cout << "num_keys          : " << config.num_keys << '\n';
    std::cout << "base_nodes        : [";
    for (std::size_t i = 0; i < base_nodes.size(); ++i) {
        std::cout << (i == 0 ? "" : ", ") << base_nodes[i];
    }
    std::cout << "]\n";
    std::cout << "added_node        : " << added_node << '\n';
    std::cout << "removed_node      : " << removed_node << '\n';
    std::cout << "virtual_node_opts : [";
    for (std::size_t i = 0; i < config.virtual_node_options.size(); ++i) {
        std::cout << (i == 0 ? "" : ", ") << config.virtual_node_options[i];
    }
    std::cout << "]\n";
    std::cout << "hash_function     : 32-bit FNV-1a + avalanche\n";
    std::cout << std::string(78, '-') << '\n';

    print_migration_table(result.migration_rows);

    std::cout << "\nModulo load summary\n";
    std::cout << std::string(78, '-') << '\n';
    for (const std::string& label : {"before", "after_add", "after_remove"}) {
        print_load_summary(label, result.modulo_summaries.at(label));
    }

    std::cout << "\nConsistent hash load summary\n";
    std::cout << std::string(78, '-') << '\n';
    for (int virtual_nodes : config.virtual_node_options) {
        std::cout << "\nvirtual_nodes = " << virtual_nodes << '\n';
        const auto& summaries = result.consistent_summaries.at(virtual_nodes);
        for (const std::string& label : {"before", "after_add", "after_remove"}) {
            print_load_summary(label, summaries.at(label));
        }
    }

    std::cout << "\nFirst ring positions for observation\n";
    std::cout << std::string(78, '-') << '\n';
    for (const auto& [position, node] : result.ring_positions) {
        std::cout << position << " -> " << node << '\n';
    }
}

}  // namespace consistent_hash


### 4.5 主程序和结果输出

主程序负责解析`--num-keys`、`--base-nodes`、`--virtual-nodes`、`--seed`和`--ring-limit`等参数，调用实验流程，并按“配置→迁移比例→负载摘要→环位置”的顺序输出结果。输出中的迁移表同时包含取模策略和不同虚拟节点数量的一致性哈希策略，便于直接对照。


In [ ]:
%%writefile src/03.02_extra_consistent_hash_table/src/main.cpp
#include "experiment.hpp"

#include <exception>
#include <iostream>
#include <string>

namespace {

void print_usage(const char* program) {
    std::cout << "Usage: " << program << " [options]\n"
              << "\nOptions:\n"
              << "  --num-keys N              Number of keys, default 10000\n"
              << "  --base-nodes N            Number of initial physical nodes, default 4\n"
              << "  --virtual-nodes LIST      Comma-separated counts, default 1,5,20,100\n"
              << "  --seed N                  Deterministic key generation seed, default 2026\n"
              << "  --ring-limit N            Ring positions to print, default 12\n"
              << "  --help                    Show this help message\n";
}

}  // namespace

int main(int argc, char** argv) {
    using namespace consistent_hash;

    // 先使用默认配置，再由命令行参数逐项覆盖。
    ExperimentConfig config;

    try {
        for (int i = 1; i < argc; ++i) {
            const std::string arg = argv[i];
            // 统一读取选项后的参数值，并检查是否缺少值。
            auto require_value = [&](const std::string& name) -> std::string {
                if (i + 1 >= argc) {
                    throw std::invalid_argument(name + " requires a value");
                }
                return argv[++i];
            };

            if (arg == "--num-keys") {
                config.num_keys = std::stoi(require_value(arg));
            } else if (arg == "--base-nodes") {
                config.num_base_nodes = std::stoi(require_value(arg));
            } else if (arg == "--virtual-nodes") {
                config.virtual_node_options = parse_virtual_nodes(require_value(arg));
            } else if (arg == "--seed") {
                config.seed = static_cast<std::uint32_t>(std::stoul(require_value(arg)));
            } else if (arg == "--ring-limit") {
                config.ring_observation_limit = std::stoi(require_value(arg));
            } else if (arg == "--help") {
                print_usage(argv[0]);
                return 0;
            } else {
                throw std::invalid_argument("unknown option: " + arg);
            }
        }

        // 这里生成节点名称只用于打印；run_experiment 内部会用相同规则构造实验场景。
        const auto base_nodes = make_base_nodes(config.num_base_nodes);
        const auto added_node = "node_" + std::to_string(config.num_base_nodes);
        const auto removed_node = base_nodes[1];

        // 执行实验后统一输出迁移比例、负载均衡和环位置。
        const auto result = run_experiment(config);
        print_experiment(config, result, base_nodes, added_node, removed_node);
    } catch (const std::exception& err) {
        std::cerr << "error: " << err.what() << '\n';
        std::cerr << "Use --help for usage.\n";
        return 1;
    }

    return 0;
}


### 4.6 核心源码检查

检查全部头文件和源文件是否已经写入。全部显示`OK`时，说明Notebook生成的C++工程源码完整，可以进入构建和运行阶段。


In [ ]:
required_files = [
    "include/stable_hash.hpp",
    "include/consistent_hash_ring.hpp",
    "include/metrics.hpp",
    "include/experiment.hpp",
    "src/stable_hash.cpp",
    "src/consistent_hash_ring.cpp",
    "src/metrics.cpp",
    "src/experiment.cpp",
    "src/main.cpp",
]

for relative_path in required_files:
    path = WORK_DIR / relative_path
    print(f"{relative_path}: {'OK' if path.exists() else 'MISSING'}")


---
## 5. 结果验证与性能分析

核心源码准备完成后，本节写入CMake构建配置和运行脚本，运行默认实验并保存输出。由于本实验是单机模拟，不需要NPU或多进程启动，运行结果可以直接在Notebook中复现。


### 5.1 工程构建

`CMakeLists.txt`使用C++17组织5个源文件，启用`-Wall -Wextra -pedantic`警告选项。工程只依赖C++标准库，适合在普通Linux服务器或Notebook环境中构建。


In [ ]:
%%writefile src/03.02_extra_consistent_hash_table/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)

project(consistent_hash_table_experiment LANGUAGES CXX)

set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
set(CMAKE_CXX_EXTENSIONS OFF)

add_executable(consistent_hash_table
    src/main.cpp
    src/stable_hash.cpp
    src/consistent_hash_ring.cpp
    src/metrics.cpp
    src/experiment.cpp
)

target_include_directories(consistent_hash_table PRIVATE include)

if (MSVC)
    target_compile_options(consistent_hash_table PRIVATE /W4 /utf-8)
else()
    target_compile_options(consistent_hash_table PRIVATE -Wall -Wextra -pedantic)
endif()


### 5.2 编译和运行脚本

`scripts/run.sh`在工程根目录执行CMake配置、编译和可执行文件运行三个步骤，并把Notebook传入的命令行参数继续传递给程序。脚本采用`set -euo pipefail`，任一构建或运行步骤失败都会返回非零状态。


In [ ]:
%%writefile src/03.02_extra_consistent_hash_table/scripts/run.sh
#!/usr/bin/env bash
set -euo pipefail

SCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
PROJECT_DIR="$(cd "${SCRIPT_DIR}/.." && pwd)"
BUILD_DIR="${PROJECT_DIR}/build"

cmake -S "${PROJECT_DIR}" -B "${BUILD_DIR}"
cmake --build "${BUILD_DIR}"
"${BUILD_DIR}/consistent_hash_table" "$@"


In [ ]:
!chmod +x src/03.02_extra_consistent_hash_table/scripts/run.sh
!find src/03.02_extra_consistent_hash_table -maxdepth 3 -type f | sort


### 5.3 运行默认实验

下面的命令使用默认配置完成编译和运行：10000个key、4个初始物理节点、`1,5,20,100`四组虚拟节点数量。`tee`会把终端输出同时保存到`results/notebook_result.txt`，便于后续读取。


In [ ]:
!cd src/03.02_extra_consistent_hash_table && bash scripts/run.sh | tee results/notebook_result.txt


### 5.4 运行参数对照实验

完成默认实验后，可以改变key数量、节点数量和虚拟节点配置，观察实验结论是否稳定。下面给出一个较小规模的快速实验命令，适合先检查参数解析和输出格式。


In [ ]:
!cd src/03.02_extra_consistent_hash_table && bash scripts/run.sh --num-keys 2000 --base-nodes 4 --virtual-nodes 1,5,20 --seed 2026 --ring-limit 8 | tee results/small_notebook_result.txt


### 5.5 读取实验结果

下面读取保存的输出文件，并截取前8000个字符显示。重点观察`Migration ratio comparison`、`Modulo load summary`和`Consistent hash load summary`三个部分。


In [ ]:
result_path = WORK_DIR / "results" / "notebook_result.txt"
if result_path.exists():
    print(result_path.read_text(encoding="utf-8", errors="ignore")[:8000])
else:
    print("not found:", result_path)


### 5.6 结果验证

结果验证主要从输入一致性、路由完整性和指标逻辑三个方面进行：

1. 所有策略共享相同的key集合、初始物理节点和随机种子；
2. 每个key在每种场景下都应得到一个物理节点归属，变更前后的assignment map大小一致；
3. 迁移数量不应超过key总数，迁移比例应处于$[0,1]$范围；
4. 每个负载摘要的节点数量与对应场景节点列表一致，`imbalance_ratio`应为最大负载与平均负载之比；
5. 哈希环位置按升序输出，虚拟节点数为$v$、物理节点数为$n$时，环位置数量应为$n\times v$。


### 5.7 迁移比例分析

取模分片的迁移比例通常较高，因为节点数量变化会直接改变取模基数。一致性哈希的迁移范围受新增节点或删除节点相邻环区间限制，因此通常明显低于取模基线，但具体比例取决于key位置和虚拟节点位置。

报告中的参考运行显示：取模分片在节点加入时迁移约79.86%的key，在节点删除时迁移约74.63%的key；一致性哈希在5个虚拟节点配置下，新增节点迁移约12.43%，删除节点迁移约21.47%。这些数值依赖key规模、种子、节点数量和哈希位置，Notebook重新运行时应以实际输出为准。


### 5.8 负载均衡分析

虚拟节点把一个物理节点负责的环区间分散到多个位置，通常可以降低负载标准差和`imbalance_ratio`。当虚拟节点数从1增加到20时，参考实验中的不均衡率从约1.933降至约1.064，说明虚拟节点对负载均衡有明显改善。

需要注意，迁移比例和负载均衡是两个不同维度：增加虚拟节点可能改善静态负载，但不会保证迁移比例单调下降；因此应分别比较迁移表和负载摘要，并结合具体哈希环位置解释结果。


In [ ]:
import re

output_path = WORK_DIR / "results" / "notebook_result.txt"
if output_path.exists():
    output = output_path.read_text(encoding="utf-8", errors="ignore")
    migration_lines = [
        line for line in output.splitlines()
        if line.startswith("modulo") or line.startswith("consistent_hash")
    ]
    print("Migration rows:")
    for line in migration_lines:
        print(line)
    print("\nPrinted ring positions:", len(re.findall(r"^\d+ -> node_", output, re.MULTILINE)))
else:
    print("not found:", output_path)


---
## 6. 实验总结

本实验按照实验概述、环境准备、问题分析、核心程序开发和结果验证与性能分析五个阶段，实现了基于一致性哈希环的分布式哈希表模拟。

* 稳定哈希函数为key和虚拟节点提供跨运行可复现的哈希位置。
* 取模分片通过`hash(key) % node_count`完成路由，但节点数量变化时会导致大范围key重新映射。
* 一致性哈希将物理节点扩展为虚拟节点并映射到环上，key通过顺时针后继规则定位物理节点。
* 节点加入和删除时，通过比较同一批key的前后归属统计迁移数量和迁移比例。
* 虚拟节点把物理节点负责的环区间切分得更细，通常可以改善静态负载均衡，但不保证所有指标单调优化。
* 结果分析需要同时关注迁移比例、负载标准差、不均衡率和环位置，分别评价动态维护成本与静态分布效果。

通过本实验，可以理解分布式哈希表从key哈希、节点路由到节点动态维护的完整流程，掌握一致性哈希环和虚拟节点的组织方法，并具备根据实验指标分析分片策略的能力。
